Σωστό workflow. Ο παρακάτω κώδικας:

διαβάζει το milestone_sampling_plan.csv.
ζητά από τον χρήστη επιλογή 1, 2, 3, 4 milestone.
κρατά μόνο τις γραμμές όπου add_in_milestone αντιστοιχεί στο milestone.
ψάχνει τα αντίστοιχα pdfs μέσα στον φάκελο oa_pdf.
αντιγράφει μόνο όσα υπάρχουν στον oa_pdf, στον φάκελο PDF.
προαιρετικά καθαρίζει πρώτα τον φάκελο PDF, ώστε να περιέχει μόνο τα pdfs του τρέχοντος milestone.

In [ ]:
""" Προσάρτηση του Google Drive """
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import shutil
import pandas as pd

# =========================
# ΡΥΘΜΙΣΕΙΣ ΦΑΚΕΛΩΝ
# =========================

BASE_DIR = "/content/drive/MyDrive/ΔΙΔΑΚΤΟΡΙΚΟ PhD/4o PAPER"

PLAN_CSV = os.path.join(BASE_DIR, "coverage_sampling_plan", "milestone_sampling_plan.csv")

OA_PDF_DIR = os.path.join(BASE_DIR, "oa_pdf")
TARGET_PDF_DIR = os.path.join(BASE_DIR, "PDF")

os.makedirs(TARGET_PDF_DIR, exist_ok=True)


In [ ]:
# =========================
# ΔΙΑΘΕΣΙΜΑ MILESTONES
# =========================

milestone_options = {
    "1": "v1_coverage_30pct",
    "2": "v2_coverage_50pct",
    "3": "v3_coverage_70pct",
    "4": "v4_coverage_100pct"
}

print("Διάλεξε milestone:")
for key, value in milestone_options.items():
    print(f"{key} = {value}")

choice = input("\nΔώσε επιλογή 1, 2, 3 ή 4: ").strip()

if choice not in milestone_options:
    raise ValueError("Μη έγκυρη επιλογή. Δώσε 1, 2, 3 ή 4.")

selected_milestone = milestone_options[choice]

print(f"\nΕπέλεξες: {selected_milestone}")


In [ ]:
# =========================
# ΦΟΡΤΩΣΗ PLAN
# =========================

df = pd.read_csv(PLAN_CSV)

required_cols = {"filename", "add_in_milestone"}
missing_cols = required_cols - set(df.columns)

if missing_cols:
    raise ValueError(f"Λείπουν οι στήλες από το CSV: {missing_cols}")

In [ ]:
# =========================
# ΦΙΛΤΡΑΡΙΣΜΑ PDF ΓΙΑ ΤΟ MILESTONE
# =========================

selected_df = df[df["add_in_milestone"] == selected_milestone].copy()

selected_filenames = (
    selected_df["filename"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

print(f"\nPDF που ζητούνται από το milestone: {len(selected_filenames)}")

In [ ]:
# =========================
# ΠΡΟΑΙΡΕΤΙΚΟ ΚΑΘΑΡΙΣΜΑ TARGET ΦΑΚΕΛΟΥ
# =========================

clear_target = input(
    "\nΝα καθαριστεί πρώτα ο φάκελος PDF ώστε να μείνουν μόνο τα νέα PDFs; (y/n): "
).strip().lower()

if clear_target == "y":
    removed_count = 0

    for file in os.listdir(TARGET_PDF_DIR):
        if file.lower().endswith(".pdf"):
            file_path = os.path.join(TARGET_PDF_DIR, file)
            os.remove(file_path)
            removed_count += 1

    print(f"Καθαρίστηκαν {removed_count} παλιά PDF από τον φάκελο PDF.")

In [ ]:
# =========================
# ΑΝΤΙΓΡΑΦΗ PDF
# =========================

copied = []
missing = []

for filename in selected_filenames:
    source_path = os.path.join(OA_PDF_DIR, filename)
    target_path = os.path.join(TARGET_PDF_DIR, filename)

    if os.path.exists(source_path):
        shutil.copy2(source_path, target_path)
        copied.append(filename)
    else:
        missing.append(filename)

In [ ]:
# =========================
# ΑΝΑΦΟΡΑ
# =========================

print("\n==============================")
print("ΑΠΟΤΕΛΕΣΜΑ ΑΝΤΙΓΡΑΦΗΣ")
print("==============================")

print(f"Milestone: {selected_milestone}")
print(f"Ζητήθηκαν συνολικά: {len(selected_filenames)} PDF")
print(f"Αντιγράφηκαν: {len(copied)} PDF")
print(f"Δεν βρέθηκαν στο oa_pdf: {len(missing)} PDF")

if missing:
    missing_report_path = os.path.join(
        BASE_DIR,
        f"missing_pdfs_{selected_milestone}.csv"
    )

    pd.DataFrame({"missing_filename": missing}).to_csv(
        missing_report_path,
        index=False,
        encoding="utf-8-sig"
    )

    print(f"\nΑποθηκεύτηκε λίστα με τα missing PDFs εδώ:")
    print(missing_report_path)

print("\nΟλοκληρώθηκε.")

END.